# 02. 주소 → 우편번호 조회 색인 v2 (전국)

v1(`02_build_address_index_v1.ipynb`)과 **같은 색인 로직**을 전국 우편번호 DB(16개 시도 파일, 6,459,663행)에 적용한다. SHP는 쓰지 않는다. 전국 실행은 약 15~16분(4코어 중 1개), 최대 메모리 약 0.9GB다.

**실행**: `data/notebooks`에서 실행하고, 출력 위치(`KRPC_OUT`)를 `data/processed`가 아닌 다른 폴더로 지정해야 한다.

**v1과 다른 점** (로직은 같고, 전국 규모를 위해 아래만 바꿨다)
1. `REGIONS`를 `zipcode_DB/*.txt` 전체로 바꾸고 시군구 접두어 필터를 뺐다.
2. 6.46M행을 한 번에 올리지 않고 **시도 파일 하나씩** 만든다(시군구 조각은 시군구 코드 기준이라 시도 파일 사이에 겹치지 않는다).
3. 노트북 01의 `hierarchy.json` 대신 DB에서 시군구 이름·시도를 직접 만든다(선택 지역 전용 파일이라 전국에 없다).
4. 전국 DB에는 **옛 법정동코드가 남은 행**(전북 45xxx, 전남 나주 46170, 화성시 41590 등 24행)이 있어, 새 코드로 **매핑해 합친다**. 옛 코드 판별·매핑 규칙과 근거는 QA 엑셀의 `옛코드_매핑`·`옛코드_행` 시트에 남긴다. 매핑하지 못한 코드가 있으면 중단한다.
5. 전수 대조는 시도별로 바로 수행하고, `KRPC_VERIFY_STEP=n`이면 n행에 1행만 대조한다(시간이 오래 걸릴 때). `KRPC_ONLY`로 시도 일부만 시험할 수 있다.
6. 출력이 `data/processed`이면 중단한다(저장된 결과 보호). 항상 `KRPC_OUT`을 다른 폴더로 지정한다.

In [1]:
import os, json, re, collections, warnings, time, resource
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)

RAW = Path(os.environ.get("KRPC_RAW", Path.cwd().parent / "raw"))
OUT = Path(os.environ.get("KRPC_OUT", Path.cwd().parent / "processed"))
assert OUT.resolve() != (Path.cwd().parent / "processed").resolve(), "저장된 data/processed 를 덮어쓰지 않도록 중단합니다. KRPC_OUT 을 다른 폴더로 지정하세요."
ADDR = OUT / "addr"
ADDR.mkdir(parents=True, exist_ok=True)
for old in ADDR.glob("*.json"):        # 이전 실행 잔여물 제거
    old.unlink()

# v2: 지역 설정만 전국으로 (zipcode_DB 의 시도 파일 전부, 시군구 접두어 필터 없음)
REGIONS = [{"key": p.stem, "sido": p.stem, "db": p.name, "sigungu_prefix": None} for p in sorted((RAW / "zipcode_DB").glob("*.txt"))]
ONLY = [s for s in os.environ.get("KRPC_ONLY", "").split(",") if s]      # 시험 실행용 (파일 이름, 확장자 제외)
if ONLY:
    REGIONS = [r for r in REGIONS if r["key"] in ONLY]
VERIFY_STEP = int(os.environ.get("KRPC_VERIFY_STEP", "1"))               # 1 = 전수 대조
BASE_DATE = "2026-10-08"
NOTICE_DATE = "2026-10-13"
T0 = time.time()
def rss_gb(): return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024
print("RAW =", RAW.resolve()); print("OUT =", OUT.resolve()); print("대상 파일", len(REGIONS), "개 / VERIFY_STEP", VERIFY_STEP)

RAW = /home/user/kr-postcode/data/raw
OUT = /tmp/claude-0/-home-user-kr-postcode/3cd683f9-9a56-554d-88fe-73d934735cc7/scratchpad/out/full2
대상 파일 16 개 / VERIFY_STEP 1


## 1. 우편번호 DB 읽기 (시도 파일 하나씩)
한 줄 = 건물 1건. 먼저 모든 파일에서 법정동 표를 뽑아 옛 법정동코드를 찾아 새 코드로 매핑하고, 새 코드 기준으로 시군구 이름·코드를 만든다(가벼운 읽기). 조각 만들기 단계에서 시도 파일을 다시 하나씩 읽는다.

In [2]:
DB_COLS = ["우편번호", "시도", "시군구", "읍면", "도로명", "지하여부", "건물번호본번", "건물번호부번",
           "시군구용건물명", "법정동코드", "법정동명", "리명", "산여부", "지번본번", "지번부번"]
LEGACY_MAP = {}          # 옛 법정동코드(10자리) → 새 법정동코드(10자리). 아래 0차 읽기에서 채운다.

def read_db(r):
    d = pd.read_csv(RAW / "zipcode_DB" / r["db"], sep="|", encoding="utf-8-sig", dtype=str, usecols=DB_COLS, keep_default_na=False)
    d["법정동코드_원본"] = d["법정동코드"]
    d["법정동코드"] = d["법정동코드"].map(LEGACY_MAP).fillna(d["법정동코드"])      # 옛 코드 → 새 코드 (매핑 목록은 QA 엑셀 '옛코드_매핑')
    d["code"] = d["법정동코드"].str[:5]
    d["emd"] = d["읍면"]
    d["label"] = np.where(d["법정동명"] != "", d["법정동명"], (d["읍면"] + " " + d["리명"]).str.strip())
    for c in ["건물번호본번", "건물번호부번", "지번본번", "지번부번", "지하여부", "산여부"]:
        assert d[c].str.fullmatch(r"\d+").all(), f"{r['db']} {c} 숫자가 아닌 값"
        d[c] = d[c].astype(int)
    assert d["우편번호"].str.fullmatch(r"\d{5}").all()
    return d

# 0차 읽기: 법정동(10자리)별 이름·건수 표 → 옛 법정동코드 찾기
file_rows = {}; emd_parts = []
for r in REGIONS:
    d = pd.read_csv(RAW / "zipcode_DB" / r["db"], sep="|", encoding="utf-8-sig", dtype=str, usecols=["우편번호", "시도", "시군구", "읍면", "법정동코드", "법정동명", "리명"], keep_default_na=False)
    file_rows[r["key"]] = len(d)
    emd_parts.append(d.groupby(["시도", "시군구", "읍면", "법정동명", "리명", "법정동코드"]).agg(행수=("우편번호", "size"), 우편번호=("우편번호", lambda s: ",".join(sorted(set(s))[:5]))).reset_index())
    del d
emd_tbl = pd.concat(emd_parts, ignore_index=True)
emd_tbl["c5"] = emd_tbl["법정동코드"].str[:5]

# 옛 코드의 정의 (두 경우만 옛 코드로 본다)
#  (가) 같은 (시도, 시군구) 이름에 코드가 둘 이상일 때, 행 수가 적은 쪽 코드  (예: 전북 군산시 45130 ↔ 52130)
#  (나) 구가 생긴 시의 구 없는 상위 코드  (예: '화성시'가 '화성시 만세구'… 로 나뉜 뒤에 남은 41590)
name_codes = collections.defaultdict(dict)
for (sido, sgg, c5), n in emd_tbl.groupby(["시도", "시군구", "c5"])["행수"].sum().items(): name_codes[(sido, sgg)][c5] = int(n)
dominant5 = {k: max(v, key=v.get) for k, v in name_codes.items()}
sgg_by_sido = collections.defaultdict(set)
for (sido, sgg) in name_codes: sgg_by_sido[sido].add(sgg)
legacy5 = {}
for k, v in name_codes.items():
    for c5 in v:
        if len(v) > 1 and c5 != dominant5[k]: legacy5[c5] = "(가) 같은 시군구 이름에 소수 코드"
for (sido, sgg), v in name_codes.items():
    if sgg and any(o.startswith(sgg + " ") for o in sgg_by_sido[sido]):
        for c5 in v: legacy5.setdefault(c5, "(나) 구가 생긴 시의 상위 코드")

# 새 코드 쪽 색인
nonleg = emd_tbl[~emd_tbl["c5"].isin(legacy5)]
new10_key = {r.법정동코드: (r.시도, r.읍면, r.법정동명, r.리명) for r in nonleg.itertuples(index=False)}
by_key = collections.defaultdict(dict)                       # (시도, 읍면, 법정동명, 리명) → {새 10자리: (시군구, 행수)}
for r in nonleg.itertuples(index=False): by_key[(r.시도, r.읍면, r.법정동명, r.리명)][r.법정동코드] = (r.시군구, int(r.행수))

REASON1 = "옛 시도 코드가 남은 행(원본 DB에 새 시도 코드로 갱신되지 않은 것으로 추정, 원인은 우체국에 확인하지 못함). 같은 (시도, 시군구) 이름의 새 코드로 앞 5자리를 바꾼 10자리가 새 코드 행에 같은 읍면·법정동명·리명으로 있어 연결"
REASON2 = "구가 생기기 전 시 단위 코드가 남은 행(추정). 시도·읍면·법정동명·리명이 같고 시군구 이름이 '시 이름 + 구'인 새 코드가 하나뿐이어서 연결"
map_rows = []
leg = emd_tbl[emd_tbl["c5"].isin(legacy5)].groupby("법정동코드").agg(시도=("시도", "first"), 시군구=("시군구", "first"), 읍면=("읍면", "first"), 법정동명=("법정동명", "first"),
        리명=("리명", "first"), 행수=("행수", "sum"), 우편번호=("우편번호", lambda s: ",".join(sorted(set(",".join(s).split(","))))), c5=("c5", "first")).reset_index()
for r in leg.itertuples(index=False):
    key = (r.시도, r.읍면, r.법정동명, r.리명); new10 = None; rule = None; reason = None; n_new = None
    new5 = dominant5[(r.시도, r.시군구)]
    cand = new5 + r.법정동코드[5:]
    if new5 != r.c5 and new10_key.get(cand) == key:
        new10, rule, reason, n_new = cand, "규칙1 시도·시군구 코드 교체", REASON1, by_key[key][cand][1]
    else:
        cs = {c: v for c, v in by_key.get(key, {}).items() if v[0] == r.시군구 or v[0].startswith(r.시군구 + " ")}
        if len(cs) == 1:
            (new10, (_, n_new)), = cs.items(); rule, reason = "규칙2 읍면·리 이름 일치(구 신설)", REASON2
    map_rows.append({"원래 법정동코드": r.법정동코드, "새 법정동코드": new10, "원래 시군구코드": r.c5, "새 시군구코드": new10[:5] if new10 else None,
                     "시도": r.시도, "시군구(원본 표기)": r.시군구, "읍면": r.읍면, "법정동명": r.법정동명, "리명": r.리명, "행수": int(r.행수), "우편번호": r.우편번호,
                     "옛 코드로 본 이유": legacy5[r.c5], "적용 규칙": rule, "이유": reason, "새 코드 행 수(근거)": n_new, "상태": "매핑" if new10 else "미매핑"})
legacy_map_df = pd.DataFrame(map_rows)
LEGACY_MAP.update({r["원래 법정동코드"]: r["새 법정동코드"] for r in map_rows if r["새 법정동코드"]})
n_leg_rows = int(legacy_map_df["행수"].sum()); n_unmapped = int((legacy_map_df["상태"] == "미매핑").sum())
print(f"옛 법정동코드 {len(legacy_map_df)}개({n_leg_rows}행, 시군구 코드 {len(legacy5)}개) → 매핑 {len(LEGACY_MAP)}개, 미매핑 {n_unmapped}개")
print(legacy_map_df["적용 규칙"].value_counts(dropna=False).to_string())
assert n_unmapped == 0, "매핑하지 못한 옛 코드가 있습니다 (QA 엑셀 확인)"

# 1차 읽기: 새 코드 기준 시군구 이름·시도, (시도, 시군구) → 코드, 법정동 표
sgg_cnt = collections.Counter(); label_cnt = collections.defaultdict(collections.Counter); sido_cnt = collections.defaultdict(collections.Counter)
area_parts = []
for r in REGIONS:
    d = pd.read_csv(RAW / "zipcode_DB" / r["db"], sep="|", encoding="utf-8-sig", dtype=str, usecols=["시도", "시군구", "읍면", "법정동코드", "법정동명", "리명"], keep_default_na=False)
    d["법정동코드"] = d["법정동코드"].map(LEGACY_MAP).fillna(d["법정동코드"])
    d["code"] = d["법정동코드"].str[:5]
    d["sg"] = np.where(d["시군구"] != "", d["시군구"], d["시도"])
    for k, n in d.groupby(["시도", "시군구", "code"]).size().items(): sgg_cnt[k] += int(n)
    for (c, s), n in d.groupby(["code", "sg"]).size().items(): label_cnt[c][s] += int(n)
    for (c, s), n in d.groupby(["code", "시도"]).size().items(): sido_cnt[c][s] += int(n)
    d["jkey_emd"] = np.where(d["읍면"] != "", d["읍면"], d["법정동명"])
    area_parts.append(d.drop_duplicates(["code", "법정동코드"])[["code", "법정동코드", "jkey_emd", "리명"]])
    del d
CODE_NAME = {c: v.most_common(1)[0][0] for c, v in label_cnt.items()}
CODE_SIDO = {c: v.most_common(1)[0][0] for c, v in sido_cnt.items()}
area_tbl = pd.concat(area_parts, ignore_index=True).drop_duplicates(["code", "법정동코드"])

# (시도, 시군구) → 코드 : 범위주소 파일에 코드가 없어 이 표로 연결한다. 세종은 시군구가 비어 있다.
# 옛 코드를 합친 뒤에는 코드가 하나뿐이어야 한다(v1과 같은 가정).
cand = collections.defaultdict(dict)
for (sido, sgg, code_), n in sgg_cnt.items(): cand[(sido, sgg)][code_] = n
multi = {k: v for k, v in cand.items() if len(v) > 1}
assert not multi, f"옛 코드를 합친 뒤에도 (시도, 시군구)가 코드 여러 개: {list(multi.items())[:3]}"
KEY2CODE = {k: next(iter(v)) for k, v in cand.items()}
print(f"DB {sum(file_rows.values()):,d}행, 파일 {len(file_rows)}개, 시군구 코드 {len(CODE_NAME)}개, (시도, 시군구) 키 {len(KEY2CODE)}개")
print(f"0·1차 읽기 경과 {time.time()-T0:.0f}s, 최대 메모리 {rss_gb():.1f}GB")


옛 법정동코드 18개(24행, 시군구 코드 9개) → 매핑 18개, 미매핑 0개
적용 규칙
규칙1 시도·시군구 코드 교체        15
규칙2 읍면·리 이름 일치(구 신설)     3


DB 6,459,663행, 파일 16개, 시군구 코드 256개, (시도, 시군구) 키 257개
0·1차 읽기 경과 34s, 최대 메모리 0.3GB


## 2. 범위주소 읽기 (참고자료)
범위종류: 0 해당 주소 1건, 1 홀수 전체, 2 짝수 전체, 3 전체. 범위 안의 건물은 모두 같은 우편번호라는 뜻이다.

In [3]:
rg = pd.read_csv(RAW / "rangeaddr" / "20261013_도로명범위.txt", sep="|", encoding="utf-8-sig", dtype=str, keep_default_na=False)
rg.columns = ["zip", "sido", "sido_en", "sgg", "sgg_en", "emd", "emd_en", "road", "road_en", "under", "sb", "sbu", "eb", "ebu", "kind"]
rg_all = len(rg)
rg["code"] = [KEY2CODE.get(k) for k in zip(rg["sido"], rg["sgg"])]
rg = rg[rg["code"].notna()].copy()
# 끝번호가 빈 줄은 번호 1개만 가리키는 줄(범위종류 0 등)이라 끝=시작으로 둔다
rg_blank_end = int((rg["eb"] == "").sum())
rg["eb"] = np.where(rg["eb"] == "", rg["sb"], rg["eb"]); rg["ebu"] = np.where(rg["ebu"] == "", rg["sbu"], rg["ebu"])
print("도로명범위 끝번호 빈 줄", rg_blank_end, "/ 범위종류별", rg.loc[rg["kind"].astype(int) >= 0, "kind"].value_counts().to_dict())
for c in ["under", "sb", "sbu", "eb", "ebu", "kind"]:
    rg[c] = rg[c].astype(int)
print(f"도로명범위 {rg_all:,d}행 중 대상 지역 {len(rg):,d}행")

jr = pd.read_csv(RAW / "rangeaddr" / "20261013_지번범위.txt", sep="|", encoding="utf-8-sig", dtype=str, keep_default_na=False)
jr.columns = ["zip", "sido", "sido_en", "sgg", "sgg_en", "emd", "emd_en", "ri", "san", "hdong", "sb", "sbu", "eb", "ebu"]
jr_all = len(jr)
jr["code"] = [KEY2CODE.get(k) for k in zip(jr["sido"], jr["sgg"])]
jr = jr[jr["code"].notna()].copy()
jr_blank_end = int((jr["eb"] == "").sum())
jr["eb"] = np.where(jr["eb"] == "", jr["sb"], jr["eb"]); jr["ebu"] = np.where(jr["ebu"] == "", jr["sbu"], jr["ebu"])
print("지번범위 끝번호 빈 줄", jr_blank_end)
for c in ["san", "sb", "sbu", "eb", "ebu"]:
    jr[c] = jr[c].astype(int)
print(f"지번범위 {jr_all:,d}행 중 대상 지역 {len(jr):,d}행")

# 지번범위의 (읍면동, 리명)을 DB의 법정동 코드에 연결한다.
JKEY2BJD = collections.defaultdict(set)
for code, bjd, e, ri in area_tbl.itertuples(index=False):
    JKEY2BJD[(code, e, ri)].add(bjd)
jr["bjds"] = [JKEY2BJD.get(k) for k in zip(jr["code"], jr["emd"], jr["ri"])]
jr_unmatched = jr[jr["bjds"].isna()]
print(f"지번범위 법정동 연결 실패 {len(jr_unmatched):,d}행 ({len(jr_unmatched)/len(jr):.2%})")

도로명범위 끝번호 빈 줄 40568 / 범위종류별 {'3': 165232, '1': 53215, '2': 53072, '0': 40568}
도로명범위 312,087행 중 대상 지역 312,087행


지번범위 끝번호 빈 줄 73819
지번범위 253,726행 중 대상 지역 253,726행


지번범위 법정동 연결 실패 0행 (0.00%)


## 3. 시군구별 조각 만들기
숫자는 모두 정수 배열로 납작하게 펼쳐 크기를 줄인다. 형식 (자세한 내용은 `data/README.md`):
- `zips` 우편번호 목록(이하 번호는 이 목록의 위치), `areas` `[법정동코드, 이름, 읍면]`
- `roads[도로명].b` : `본번, 부번, 지하, 우편번호, 지역` 5개씩 / `roads[도로명].r` : `시작본, 시작부, 끝본, 끝부, 지하, 종류, 우편번호, 읍면` 8개씩
- `bn` 건물명 `[이름, 도로명, 본번, 부번, 우편번호, 지역]`
- `j[지역]` : `산, 본번, 부번, 우편번호` 4개씩 / `jr[지역]` : `산, 시작본, 시작부, 끝본, 끝부, 우편번호` 6개씩

## 4. 조회 함수 (사이트 `addr.ts`와 같은 규칙)
v1 5절의 함수를 그대로 가져왔다. 시도별로 조각을 만든 직후 이 함수로 대조한다.

In [4]:
def road_lookup(sh, road, bon, bu, under, area_emd=None):
    """returns (level, [zips])  level: 'exact' | 'range' | None"""
    r = sh["roads"].get(road)
    if not r:
        return None, []
    b = r.get("b", [])
    hit = sorted({sh["zips"][b[i + 3]] for i in range(0, len(b), 5) if b[i] == bon and b[i + 1] == bu and b[i + 2] == under})
    if hit:
        return "exact", hit
    rr = r.get("r", [])
    zs = set()
    for i in range(0, len(rr), 8):
        sb, sbu, eb, ebu, u, kind, z, e = rr[i:i + 8]
        if u != under:
            continue
        if area_emd is not None and sh["emds"][e] != area_emd:
            continue
        if kind == 1 and bon % 2 == 0: continue
        if kind == 2 and bon % 2 == 1: continue
        if (sb, sbu) <= (bon, bu) <= (eb, ebu):
            zs.add(sh["zips"][z])
    return ("range", sorted(zs)) if zs else (None, [])

def jibun_lookup(sh, area_i, san, bon, bu):
    j = sh["j"].get(str(area_i), [])
    hit = sorted({sh["zips"][j[i + 3]] for i in range(0, len(j), 4) if j[i] == san and j[i + 1] == bon and j[i + 2] == bu})
    if hit:
        return "exact", hit
    jr_ = sh["jr"].get(str(area_i), [])
    zs = {sh["zips"][jr_[i + 5]] for i in range(0, len(jr_), 6) if jr_[i] == san and (jr_[i + 1], jr_[i + 2]) <= (bon, bu) <= (jr_[i + 3], jr_[i + 4])}
    return ("range", sorted(zs)) if zs else (None, [])

## 5. 시도별 조각 만들기 + 전수 대조
시도 파일 하나를 읽어 시군구 조각을 만들고, **조각 파일을 다시 읽어** DB의 건물을 조회해 원래 우편번호가 나오는지 대조한다(v1 3·5절과 같은 로직).

In [5]:
def flat(rows):
    return [int(v) for row in rows for v in row]

stats, dup_dropped = [], 0
remapped_parts = []
res = collections.Counter(); bad_road, bad_range = [], []; multi_j_n = 0; multi_j = []; n_checked = 0
sido_time = []
for region in REGIONS:
    t_s = time.time()
    db = read_db(region)
    rm = db[db["법정동코드_원본"] != db["법정동코드"]]
    if len(rm): remapped_parts.append(rm[["우편번호", "시도", "시군구", "읍면", "법정동명", "리명", "도로명", "건물번호본번", "건물번호부번", "법정동코드_원본", "법정동코드"]])
    t_read = time.time() - t_s
    for code, g in db.groupby("code"):
        zips = sorted(set(g["우편번호"]) | set(rg.loc[rg["code"] == code, "zip"]) | set(jr.loc[jr["code"] == code, "zip"]))
        zi = {z: i for i, z in enumerate(zips)}
        areas = g.drop_duplicates("법정동코드").sort_values("법정동코드")[["법정동코드", "label", "emd"]]
        ai = {b: i for i, b in enumerate(areas["법정동코드"])}
        emds = sorted(set(areas["emd"]) | set(rg.loc[rg["code"] == code, "emd"]))
        ei = {e: i for i, e in enumerate(emds)}

        roads = {}
        bld = g[["도로명", "건물번호본번", "건물번호부번", "지하여부", "우편번호", "법정동코드"]].drop_duplicates()
        dup_dropped += len(g) - len(bld)
        for road, h in bld.sort_values(["도로명", "지하여부", "건물번호본번", "건물번호부번"]).groupby("도로명", sort=True):
            roads.setdefault(road, {})["b"] = flat(zip(h["건물번호본번"], h["건물번호부번"], h["지하여부"], h["우편번호"].map(zi), h["법정동코드"].map(ai)))
        for road, h in rg[rg["code"] == code].groupby("road", sort=True):
            roads.setdefault(road, {})["r"] = flat(zip(h["sb"], h["sbu"], h["eb"], h["ebu"], h["under"], h["kind"], h["zip"].map(zi), h["emd"].map(ei)))

        bn = g[g["시군구용건물명"] != ""][["시군구용건물명", "도로명", "건물번호본번", "건물번호부번", "우편번호", "법정동코드"]].drop_duplicates()
        bn_rows = [[n, r, int(b), int(u), zi[z], ai[a]] for n, r, b, u, z, a in bn.itertuples(index=False)]

        jb = g[["법정동코드", "산여부", "지번본번", "지번부번", "우편번호"]].drop_duplicates().sort_values(["법정동코드", "산여부", "지번본번", "지번부번"])
        j = {str(ai[b]): flat(zip(h["산여부"], h["지번본번"], h["지번부번"], h["우편번호"].map(zi))) for b, h in jb.groupby("법정동코드")}
        jrr = {}
        for _, row in jr[(jr["code"] == code) & jr["bjds"].notna()].iterrows():
            for b in row["bjds"]:
                jrr.setdefault(str(ai[b]), []).extend([row["san"], row["sb"], row["sbu"], row["eb"], row["ebu"], zi[row["zip"]]])

        shard = {"v": 1, "code": code, "name": CODE_NAME[code], "sido": CODE_SIDO[code], "base": BASE_DATE,
                 "zips": zips, "emds": emds, "areas": areas.values.tolist(), "roads": roads, "bn": bn_rows, "j": j, "jr": jrr}
        p = ADDR / f"{code}.json"
        p.write_text(json.dumps(shard, ensure_ascii=False, separators=(",", ":"), allow_nan=False), encoding="utf-8")
        stats.append({"code": code, "name": CODE_NAME[code], "건물행": len(g), "도로수": len(roads), "건물명": len(bn_rows),
                      "지번 지역": len(j), "KB": round(p.stat().st_size / 1024, 1)})

    t_build = time.time() - t_s - t_read

    # ---- 대조: 방금 쓴 조각을 다시 읽는다
    codes_here = sorted(db["code"].unique())
    shards = {c: json.loads((ADDR / f"{c}.json").read_text(encoding="utf-8")) for c in codes_here}
    AI = {c: {a[0]: i for i, a in enumerate(shards[c]["areas"])} for c in codes_here}
    RANGE_ONLY = {c: {**shards[c], "roads": {k: {"r": v.get("r", [])} for k, v in shards[c]["roads"].items()}} for c in codes_here}
    chk = db.iloc[::VERIFY_STEP]
    n_checked += len(chk)
    for row in chk.itertuples(index=False):
        sh = shards[row.code]
        lv, zs = road_lookup(sh, row.도로명, row.건물번호본번, row.건물번호부번, row.지하여부)
        if lv == "exact" and row.우편번호 in zs:
            res["도로명 건물 일치"] += 1
            if len(zs) > 1: res["도로명 일치 중 우편번호 둘 이상"] += 1
        else:
            res["도로명 실패"] += 1; bad_road.append((row.code, row.도로명, row.건물번호본번, row.건물번호부번, row.우편번호, str(zs)))
        lvr, zr = road_lookup(RANGE_ONLY[row.code], row.도로명, row.건물번호본번, row.건물번호부번, row.지하여부, row.emd)
        if lvr is None:
            res["범위: 해당 범위 없음"] += 1
        elif zr == [row.우편번호]:
            res["범위: 같은 우편번호"] += 1
        elif row.우편번호 in zr:
            res["범위: 후보 여럿(정답 포함)"] += 1
        else:
            res["범위: 다른 우편번호"] += 1
            if len(bad_range) < 5000: bad_range.append((row.code, row.도로명, row.건물번호본번, row.건물번호부번, row.우편번호, str(zr)))
        lj, zj = jibun_lookup(sh, AI[row.code][row.법정동코드], row.산여부, row.지번본번, row.지번부번)
        if lj == "exact" and row.우편번호 in zj:
            res["지번 건물 일치"] += 1
            if len(zj) > 1:
                multi_j_n += 1
                if len(multi_j) < 50000: multi_j.append((row.code, row.법정동코드, row.산여부, row.지번본번, row.지번부번, ",".join(zj)))
        else:
            res["지번 실패"] += 1
    t_chk = time.time() - t_s - t_read - t_build
    sido_time.append({"파일": region["key"], "행수": len(db), "읽기 s": round(t_read, 1), "조각 만들기 s": round(t_build, 1), "대조 s": round(t_chk, 1), "대조 행": len(chk)})
    print(f"{region['key']:10s} {len(db):>9,d}행  읽기 {t_read:5.1f}s  조각 {t_build:6.1f}s  대조 {t_chk:7.1f}s ({len(chk):,d}행)  경과 {time.time()-T0:6.0f}s  최대메모리 {rss_gb():.1f}GB", flush=True)
    del db, shards, RANGE_ONLY, chk

st = pd.DataFrame(stats); st_time = pd.DataFrame(sido_time)
print("조각", len(st), "개, 합계 KB", round(st["KB"].sum()), " / 중복 제거된 건물행", dup_dropped)
summary = pd.DataFrame([{"항목": k, "건수": v, "비율": round(v / n_checked, 5)} for k, v in sorted(res.items())])
print(summary.to_string(index=False))
assert res["도로명 실패"] == 0, "도로명 전수 대조 실패 있음"
assert res["지번 실패"] == 0, "지번 전수 대조 실패 있음"

강원특별자치도      372,642행  읽기   2.1s  조각   13.5s  대조    23.8s (372,642행)  경과     75s  최대메모리 0.4GB


경기도        1,038,808행  읽기   5.6s  조각   43.8s  대조    93.9s (1,038,808행)  경과    218s  최대메모리 0.9GB


경상남도         662,786행  읽기   3.3s  조각   25.3s  대조    37.1s (662,786행)  경과    284s  최대메모리 0.9GB


경상북도         729,308행  읽기   3.1s  조각   28.0s  대조    37.8s (729,308행)  경과    353s  최대메모리 0.9GB


대구광역시        229,005행  읽기   0.9s  조각    7.1s  대조    53.1s (229,005행)  경과    414s  최대메모리 0.9GB


...

전북특별자치도      441,652행  읽기   2.0s  조각   19.4s  대조    23.9s (441,652행)  경과    871s  최대메모리 0.9GB


제주특별자치도      159,970행  읽기   0.8s  조각    6.4s  대조    17.7s (159,970행)  경과    896s  최대메모리 0.9GB


충청남도         501,703행  읽기   2.5s  조각   19.1s  대조    22.4s (501,703행)  경과    940s  최대메모리 0.9GB


충청북도         342,348행  읽기   1.6s  조각   16.7s  대조    17.3s (342,348행)  경과    975s  최대메모리 0.9GB


조각 256 개, 합계 KB 203334  / 중복 제거된 건물행 33787
                항목      건수  비율
         도로명 건물 일치 6459663 1.0
도로명 일치 중 우편번호 둘 이상      30 0.0
       범위: 같은 우편번호 6459643 1.0
      범위: 해당 범위 없음       1 0.0
  범위: 후보 여럿(정답 포함)      19 0.0
          지번 건물 일치 6459663 1.0


## 4. 이름 색인 (names.json)
입력에서 지역을 알아내는 용도다. 시군구 이름, 도로명, 읍면동·리 이름이 어느 시군구에 있는지 적는다.

In [6]:
codes = sorted(CODE_NAME)
cidx = {c: i for i, c in enumerate(codes)}
road_map, area_map = collections.defaultdict(set), collections.defaultdict(set)
for c in codes:
    sh = json.loads((ADDR / f"{c}.json").read_text(encoding="utf-8"))
    for r in sh["roads"]:
        road_map[r].add(cidx[c])
    for _, label, emd in sh["areas"]:
        for t in set(label.split()) | {label}:
            area_map[t].add(cidx[c])
names = {"v": 1, "c": [[c, CODE_NAME[c], CODE_SIDO[c]] for c in codes],
         "r": {k: sorted(v) for k, v in sorted(road_map.items())},
         "a": {k: sorted(v) for k, v in sorted(area_map.items())}}
(ADDR / "names.json").write_text(json.dumps(names, ensure_ascii=False, separators=(",", ":"), allow_nan=False), encoding="utf-8")
print("names.json KB", round((ADDR / "names.json").stat().st_size / 1024, 1), "/ 도로명", len(names["r"]), "/ 지역 이름", len(names["a"]))

names.json KB 3931.3 / 도로명 138135 / 지역 이름 26029


## 6. 점검 엑셀 저장
한 파일에 시트별로 담는다(v1과 같은 시트 + 측정용 시트).

In [7]:
remapped_df = (pd.concat(remapped_parts, ignore_index=True) if remapped_parts else pd.DataFrame()).rename(columns={"법정동코드_원본": "원래 법정동코드", "법정동코드": "새 법정동코드"})
cols = ["시군구코드", "도로명", "본번", "부번", "DB 우편번호", "조회 결과"]
limits = pd.DataFrame({"한계": [
    "범위주소(도로명범위·지번범위)는 우체국이 '참고자료'로 제공한다. 건물 DB와 어긋나는 곳이 있어 `범위 기준` 등급으로만 보여 준다.",
    "건물 DB에 없는 건물(신축·미등록)은 범위로만 조회된다. 범위도 없으면 '찾지 못함'이다.",
    "전수 대조에서 범위주소가 건물 DB와 거의 모두 같게 나오는 것은 두 자료가 같은 원천에서 만들어졌기 때문이며, 우편번호 자체가 현실과 맞는지를 독립적으로 검증한 결과는 아니다.",
    "전수 대조(도로명·지번 100%)는 '조각 파일을 만들고 다시 읽는 과정에서 값이 바뀌지 않았다'는 확인이다. 원본 DB의 정확성은 별개다.",
    "한 필지에 우편번호가 둘 이상인 지번은 모두 보여 준다 (지번 다중 우편번호 시트).",
    "건물명 검색은 DB의 '시군구용건물명'만 쓴다. 다량배달처명은 이 DB에서 모두 비어 있다.",
    "지번범위를 법정동에 연결하지 못한 행은 범위 기준 조회에서 빠진다 (지번범위 연결 실패 시트).",
    "이름 색인(names.json)은 전국 한 파일이다. 크기를 보고 분할 방식을 정한다(측정 결과 참고).",
    f"전수 대조 행 수: {n_checked:,d} (KRPC_VERIFY_STEP={VERIFY_STEP}).",
    f"옛 법정동코드가 남은 {n_leg_rows}행({len(legacy_map_df)}개 코드)은 새 코드로 매핑해 합쳤다(옛코드_매핑·옛코드_행 시트). 원인은 우체국에 확인하지 못했다.",
    f"기준일 {BASE_DATE}, 고시일 {NOTICE_DATE}. 이후 변경분(변경분 DB)은 반영하지 않았다.",
]})
xl = OUT / "qa" / "address_index_qa_v2.xlsx"
xl.parent.mkdir(exist_ok=True, parents=True)
with pd.ExcelWriter(xl, engine="openpyxl") as w:
    summary.to_excel(w, sheet_name="전수대조_요약", index=False)
    st.to_excel(w, sheet_name="시군구별_조각", index=False)
    pd.DataFrame(bad_range[:5000], columns=cols).to_excel(w, sheet_name="범위_다른우편번호", index=False)
    pd.DataFrame(multi_j, columns=["시군구코드", "법정동코드", "산", "본번", "부번", "우편번호들"]).to_excel(w, sheet_name="지번_다중우편번호", index=False)
    jr_unmatched.drop(columns=["bjds"]).head(5000).to_excel(w, sheet_name="지번범위_연결실패", index=False)
    legacy_map_df.to_excel(w, sheet_name="옛코드_매핑", index=False)
    remapped_df.to_excel(w, sheet_name="옛코드_행", index=False)
    st_time.to_excel(w, sheet_name="시도별_시간", index=False)
    limits.to_excel(w, sheet_name="데이터_한계", index=False)
meta = {"built_from": {"base_date": BASE_DATE, "notice_date": NOTICE_DATE}, "sigungu": len(codes), "building_rows": int(sum(file_rows.values())), "verified_rows": int(n_checked), "verify_step": VERIFY_STEP, "multi_zip_jibun_rows": int(multi_j_n), "legacy_codes_remapped": int(len(LEGACY_MAP)), "legacy_rows_remapped": int(n_leg_rows), "elapsed_s": round(time.time() - T0), "max_rss_gb": round(rss_gb(), 2),
        "road_names": len(names["r"]), "shard_kb_total": round(float(st["KB"].sum()), 1),
        "results": {k: int(v) for k, v in res.items()}, "jibun_range_unmatched_rows": int(len(jr_unmatched))}
(ADDR / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(meta, ensure_ascii=False, indent=1))

{
 "built_from": {
  "base_date": "2026-10-08",
  "notice_date": "2026-10-13"
 },
 "sigungu": 256,
 "building_rows": 6459663,
 "verified_rows": 6459663,
 "verify_step": 1,
 "multi_zip_jibun_rows": 6458,
 "legacy_codes_remapped": 18,
 "legacy_rows_remapped": 24,
 "elapsed_s": 982,
 "max_rss_gb": 0.86,
 "road_names": 138135,
 "shard_kb_total": 203334.4,
 "results": {
  "도로명 건물 일치": 6459663,
  "범위: 같은 우편번호": 6459643,
  "지번 건물 일치": 6459663,
  "범위: 후보 여럿(정답 포함)": 19,
  "도로명 일치 중 우편번호 둘 이상": 30,
  "범위: 해당 범위 없음": 1
 },
 "jibun_range_unmatched_rows": 0
}
